# 06 - Fluxo de decisao automatizado (LangGraph)

Demonstra o grafo completo (`src/assistant/graph.py`):

```
receber dados -> consultar prontuario -> checar exames pendentes
  -> rodar modelo de predicao de AVC -> sugerir conduta (LLM + RAG)
  -> aplicar guardrails -> [condicional] emitir alerta -> log de auditoria
```

Rodamos 3 cenarios: paciente de alto risco (deve alertar), paciente de baixo risco com exame pendente (nao deve alertar) e um caso onde o guardrail de prescricao direta e acionado (deve alertar mesmo com risco baixo).

In [1]:
import os
import sys
from pathlib import Path

if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    ROOT = Path('/content/drive/MyDrive/stroke-prediction', force_remount=True)
    sys.path.insert(0, str(ROOT))
    !pip install -r "{ROOT / 'requirements.txt'}"
    print("Rodando no Google Colab")
else:
    ROOT = Path('..').resolve()
    sys.path.insert(0, str(ROOT))
    print("Rodando Localmente")

from src.assistant.graph import run_flow
from src.assistant.llm_backend import get_generate_fn
from src.assistant.patient_db import build_patient_db
from src.assistant.retriever import build_vectorstore
from src.security.audit_log import read_audit_log

build_patient_db()
vectorstore = build_vectorstore()
generate_fn = get_generate_fn()

Mounted at /content/drive


/tmp/ipykernel_11676/2586878124.py:8: DeprecationWarning: support for supplying keyword arguments to pathlib.PurePath is deprecated and scheduled for removal in Python 3.14
  ROOT = Path('/content/drive/MyDrive/stroke-prediction', force_remount=True)


Rodando no Google Colab


/content/drive/MyDrive/stroke-prediction/src/assistant/retriever.py:68: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name=embedding_model_name)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


## Cenario 1 — paciente de alto risco (id 9046)

In [2]:
result_high_risk = run_flow(
    patient_id=9046,
    question='Quais os proximos passos para este paciente?',
    vectorstore=vectorstore,
    generate_fn=generate_fn,
)
print('Risco de AVC:', result_high_risk['stroke_risk'])
print('Alerta emitido?', result_high_risk.get('alert'), '-', result_high_risk.get('alert_reason'))
print('Resposta:', result_high_risk['response'][:300])

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Model loaded: /content/drive/MyDrive/stroke-prediction/results/logistic_regression.joblib
Loaded dataset: 5110 rows, 12 columns
Loaded dataset: 5110 rows, 12 columns


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Risco de AVC: {'prediction': 1, 'probability': 0.7694777304028538}
Alerta emitido? True - risco de AVC alto (probabilidade=0.77)
Resposta: Resposta:

Para este paciente, o próximo passo seria verificar seu status atual através de um exame físico completo e realização de testes adicionais conforme necessário. Em seguida, o médico responsável deverá avaliar os resultados e tomar decisões baseadas nos critérios de elegibilidade para a tro


## Cenario 2 — paciente de baixo risco com exame pendente (id 51676)

In [4]:
result_low_risk = run_flow(
    patient_id=51676,
    question='Este paciente precisa de trombolise agora?',
    vectorstore=vectorstore,
    generate_fn=generate_fn,
)
print('Risco de AVC:', result_low_risk['stroke_risk'])
print('Exames pendentes:', result_low_risk['pending_exams'])
print('Alerta emitido?', result_low_risk.get('alert'))

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Model loaded: /content/drive/MyDrive/stroke-prediction/results/logistic_regression.joblib
Risco de AVC: {'prediction': 0, 'probability': 0.3156223334631985}
Exames pendentes: ['Perfil metabólico / IMC']
Alerta emitido? True


## Cenario 3 — guardrail aciona alerta mesmo com risco baixo

In [5]:
def unsafe_generate(prompt: str) -> str:
    # Simula uma saida de LLM que violaria a politica de nunca prescrever direto
    return 'Administre 10mg de enalapril e tome 500mg de AAS agora.'

result_guardrail = run_flow(
    patient_id=51676,
    question='O que fazer agora?',
    vectorstore=vectorstore,
    generate_fn=unsafe_generate,
)
print('Requer validacao humana?', result_guardrail['requires_human_validation'])
print('Alerta emitido?', result_guardrail.get('alert'), '-', result_guardrail.get('alert_reason'))
print('Resposta (com disclaimer):', result_guardrail['response'][-250:])

Model loaded: /content/drive/MyDrive/stroke-prediction/results/logistic_regression.joblib
Requer validacao humana? True
Alerta emitido? True - guardrail de prescrição direta acionado
Resposta (com disclaimer): Administre 10mg de enalapril e tome 500mg de AAS agora.

⚠️ Esta sugestão foi gerada por um assistente de IA e NÃO substitui o julgamento clínico. Requer validação humana por um médico responsável antes de qualquer conduta ou prescrição.


## Log de auditoria

Cada execucao do fluxo grava uma entrada em `results/audit_log.jsonl`.

In [6]:
entries = read_audit_log()
print(f'{len(entries)} entradas no log de auditoria')
for e in entries[-3:]:
    print('-', e['timestamp'], '| paciente (pseudonimizado):', e['patient_id'], '| alerta de validacao:', e['requires_human_validation'])

4 entradas no log de auditoria
- 2026-09-15T06:43:44.396521+00:00 | paciente (pseudonimizado): 03214801f88d8260 | alerta de validacao: False
- 2026-09-15T06:44:10.462828+00:00 | paciente (pseudonimizado): efec65f8318e703c | alerta de validacao: True
- 2026-09-15T06:44:18.569785+00:00 | paciente (pseudonimizado): efec65f8318e703c | alerta de validacao: True
